# Entrenamiento completo YOLO con hiperparámetros optimizados

Notebook reproducible en **Google Colab** y también ejecutable en local.

Flujo:
1. Carga del dataset
2. Conteo de clases
3. Muestra de 1 imagen por clase
4. Entrenamiento completo (100 épocas + hiperparámetros Optuna)
5. Visualización de curvas y matriz de confusión
6. Inferencia con `kortxo.jpg`


In [1]:
# Fase 0 (setup): instalación de dependencias + imports globales
# Breve: dejamos el entorno listo para entrenar YOLOv8 en Colab/local.

import sys

if 'google.colab' in sys.modules:
    !pip -q install ultralytics huggingface_hub pyyaml seaborn
else:
    # En local también intentamos instalar por reproducibilidad
    !pip -q install ultralytics huggingface_hub pyyaml seaborn

# Todos los imports del notebook en una sola celda
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from huggingface_hub import snapshot_download
from IPython.display import Image, display
from ultralytics import YOLO


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Carga del dataset
Breve: descargamos el dataset desde Hugging Face y detectamos la estructura YOLO.

In [2]:
DATASET_REPO = 'mikeldiez/kortxovision'
LOCAL_DATA_DIR = Path('./data/kortxovision').resolve()
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Descargando dataset {DATASET_REPO}...')
dataset_root = Path(
    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        local_dir=str(LOCAL_DATA_DIR),
        local_dir_use_symlinks=False,
    )
).resolve()
print('Dataset descargado en:', dataset_root)

# Detecta YAML de dataset (prioriza data.yml original del repo)
candidate_yaml = (
    list(dataset_root.rglob('data.yml'))
    + list(dataset_root.rglob('data.yaml'))
    + list(dataset_root.rglob('dataset.yml'))
    + list(dataset_root.rglob('dataset.yaml'))
)
if candidate_yaml:
    data_yaml_path = candidate_yaml[0]
    print('Usando YAML existente:', data_yaml_path)
else:
    # Solo si no existe ningún yaml/yml, lo construimos automáticamente
    print('No se encontró data.yml/data.yaml. Generando uno automáticamente...')
    yolo_root = dataset_root

    # Intento de root con carpetas images/labels
    if not (yolo_root / 'images').exists() or not (yolo_root / 'labels').exists():
        # Busca primer nivel que tenga images y labels
        for p in dataset_root.rglob('*'):
            if p.is_dir() and (p / 'images').exists() and (p / 'labels').exists():
                yolo_root = p
                break

    generated_yaml = {
        'path': str(yolo_root),
        'train': 'images/train',
        'val': 'images/validation' if (yolo_root / 'images' / 'validation').exists() else 'images/val',
        'test': 'images/test' if (yolo_root / 'images' / 'test').exists() else 'images/val',
        'names': {},  # lo rellenaremos después con el conteo real de clases
    }

    data_yaml_path = yolo_root / 'data.yaml'
    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(generated_yaml, f, sort_keys=False, allow_unicode=True)

    print('YAML generado en:', data_yaml_path)

with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print('\nContenido de data.yaml:')
print(data_cfg)

Descargando dataset mikeldiez/kortxovision...


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching ... files: 2102it [00:05, 418.06it/s]


Dataset descargado en: /home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision
Usando YAML existente: /home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision/data.yml

Contenido de data.yaml:
{'names': {0: 'destornillador', 1: 'tijeras', 2: 'tenazas', 3: 'llave_inglesa', 4: 'cinta_adhesiva', 5: 'rotulador', 6: 'cutter', 7: 'pinzas', 8: 'lima', 9: 'llave_AlenT', 10: 'llave_dinamometrica', 11: 'calibre', 12: 'nivel', 13: 'polimetro', 14: 'disco_pulido', 15: 'EPIS', 16: 'pelacables'}, 'nc': 17, 'path': '/home/mikel/datasets/kortxovision_mini', 'test': 'images/test', 'train': 'images/train', 'val': 'images/val'}


## 2) Conteo de clases
Breve: recorremos todos los labels y contamos instancias por clase para validar distribución.

In [ ]:
# Resolve root del dataset desde YAML
base_path = Path(data_cfg.get('path', data_yaml_path.parent)).resolve()
labels_root = base_path / 'labels'

if not labels_root.exists():
    # fallback: buscar cualquier carpeta labels
    all_labels = list(base_path.rglob('labels'))
    if not all_labels:
        raise FileNotFoundError(f'No se encontró carpeta labels bajo {base_path}')
    labels_root = all_labels[0]

label_files = list(labels_root.rglob('*.txt'))
print(f'Ficheros de label detectados: {len(label_files)}')

class_counter = Counter()
for lf in label_files:
    with open(lf, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:  # formato YOLO: class x_center y_center w h
                class_id = int(float(parts[0]))
                class_counter[class_id] += 1

if not class_counter:
    raise ValueError('No se encontraron anotaciones válidas en los labels.')

# Resolver nombres de clase (si vienen en YAML, perfecto; si no, genéricos)
if isinstance(data_cfg.get('names'), dict) and data_cfg['names']:
    names_map = {int(k): v for k, v in data_cfg['names'].items()}
elif isinstance(data_cfg.get('names'), list) and data_cfg['names']:
    names_map = {i: n for i, n in enumerate(data_cfg['names'])}
else:
    max_id = max(class_counter.keys())
    names_map = {i: f'class_{i}' for i in range(max_id + 1)}
    data_cfg['names'] = names_map
    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

rows = []
for cid in sorted(class_counter):
    rows.append({
        'class_id': cid,
        'class_name': names_map.get(cid, f'class_{cid}'),
        'instances': class_counter[cid],
    })

class_df = pd.DataFrame(rows)
display(class_df)
print('\nTotal clases:', class_df.shape[0])
print('Total instancias:', int(class_df['instances'].sum()))

## 3) 1 imagen por clase
Breve: seleccionamos un ejemplo por clase y dibujamos su bounding box para comprobar etiquetado.

In [ ]:
# Localiza carpeta de imágenes de train (fallbacks comunes)
train_images_candidates = [
    base_path / 'images' / 'train',
    base_path / 'train' / 'images',
]
train_images_dir = next((p for p in train_images_candidates if p.exists()), None)
if train_images_dir is None:
    # fallback amplio
    candidates = list(base_path.rglob('images/train')) + list(base_path.rglob('train/images'))
    if not candidates:
        raise FileNotFoundError('No se encontró carpeta de imágenes de train.')
    train_images_dir = candidates[0]

train_labels_candidates = [
    base_path / 'labels' / 'train',
    base_path / 'train' / 'labels',
]
train_labels_dir = next((p for p in train_labels_candidates if p.exists()), None)
if train_labels_dir is None:
    candidates = list(base_path.rglob('labels/train')) + list(base_path.rglob('train/labels'))
    if not candidates:
        raise FileNotFoundError('No se encontró carpeta de labels de train.')
    train_labels_dir = candidates[0]

# Mapea class_id -> (img_path, bbox)
one_sample_per_class = {}

for label_path in sorted(train_labels_dir.rglob('*.txt')):
    stem = label_path.stem
    img_path = None
    for ext in ('.jpg', '.jpeg', '.png', '.webp', '.bmp'):
        candidate = train_images_dir / f'{stem}{ext}'
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        continue

    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cid = int(float(parts[0]))
            if cid not in one_sample_per_class:
                x, y, w, h = map(float, parts[1:5])
                one_sample_per_class[cid] = (img_path, (x, y, w, h))

# Plot 1 imagen por clase
class_ids = sorted(one_sample_per_class.keys())
n = len(class_ids)
cols = 4
rows = (n + cols - 1) // cols
plt.figure(figsize=(5 * cols, 4 * rows))

for i, cid in enumerate(class_ids, start=1):
    img_path, (x, y, w, h) = one_sample_per_class[cid]
    img_bgr = cv2.imread(str(img_path))
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    ih, iw = img.shape[:2]
    x1 = int((x - w / 2) * iw)
    y1 = int((y - h / 2) * ih)
    x2 = int((x + w / 2) * iw)
    y2 = int((y + h / 2) * ih)

    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    class_name = names_map.get(cid, f'class_{cid}')

    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(f'{cid}: {class_name}')
    plt.axis('off')

plt.tight_layout()
plt.show()

missing_classes = set(class_counter.keys()) - set(one_sample_per_class.keys())
if missing_classes:
    print('Clases sin muestra visual encontrada:', sorted(missing_classes))

## 4) Entrenamiento completo (100 épocas + hiperparámetros Optuna)
Breve: entrenamos `yolov8n` durante 100 épocas usando los mejores hiperparámetros encontrados en la búsqueda.

In [ ]:
BEST_HYP = {
    'lr0': 0.0006144543785587475,
    'lrf': 0.18951730321390894,
    'momentum': 0.8820925971590665,
    'weight_decay': 0.0013826232179369874,
    'hsv_h': 0.009983689107917987,
    'hsv_s': 0.5599641068895281,
    'hsv_v': 0.6146901982034297,
    'degrees': 0.46450412719997725,
    'scale': 0.6252813963310069,
    'fliplr': 0.08526206184364576,
    'mosaic': 0.06505159298527952,
}

model = YOLO('yolov8n.pt')

train_results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=4,
    project='runs/detect',
    name='yolov8n_kortxovision_full_training',
    exist_ok=True,
    pretrained=True,
    seed=42,
    deterministic=True,
    **BEST_HYP,
)

best_weights = Path(train_results.save_dir) / 'weights' / 'best.pt'
print('Entrenamiento finalizado. Carpeta de resultados:', train_results.save_dir)
print('Pesos best.pt:', best_weights)


## 5) Curvas de entrenamiento y matriz de confusión
Breve: revisamos métricas globales del entrenamiento y errores entre clases.

In [ ]:
# Visualización de artefactos generados por Ultralytics
save_dir = Path(train_results.save_dir)

results_png = save_dir / 'results.png'
confusion_png = save_dir / 'confusion_matrix.png'
confusion_norm_png = save_dir / 'confusion_matrix_normalized.png'

if results_png.exists():
    print('Curvas de entrenamiento (loss/mAP):')
    display(Image(filename=str(results_png)))
else:
    print(f'No se encontró {results_png.name} en {save_dir}')

if confusion_png.exists():
    print('Matriz de confusión:')
    display(Image(filename=str(confusion_png)))
elif confusion_norm_png.exists():
    print('Matriz de confusión normalizada:')
    display(Image(filename=str(confusion_norm_png)))
else:
    print('No se encontró matriz de confusión en la carpeta de resultados.')

## 6) Inferencia con `kortxo.jpg`
Breve: comparamos el modelo base (`yolov8n.pt`) contra el modelo entrenado sobre la imagen fija `kortxo.jpg`.

In [ ]:
# Inferencia configurable sobre imagen fija fuera del dataset (kortxo.jpg)
# 1) modelo base sin fine-tuning
# 2) modelo entrenado en este notebook

# ---------------- Configurable ----------------
image_candidates = [
    Path('/content/kortxo.jpg').resolve(),
    Path('/content/drive/MyDrive/kortxo.jpg').resolve(),
    Path('/home/mikel/github/TKNIKA/kortxovision/notebooks/kortxo.jpg').resolve(),
    Path('/home/mikel/github/TKNIKA/kortxovision/kortxo.jpg').resolve(),
    Path('kortxo.jpg').resolve(),
]

BASE_MODEL_ID = 'yolov8n.pt'
# ----------------------------------------------

test_image = next((p for p in image_candidates if p.exists()), None)
if test_image is None:
    raise FileNotFoundError('No se encontró kortxo.jpg. Colócala en notebooks/kortxo.jpg, en la raíz del repo o en el cwd del notebook.')

trained_weights = Path(train_results.save_dir) / 'weights' / 'best.pt'
trained_weights = trained_weights.resolve()
if not trained_weights.exists():
    raise FileNotFoundError(f'No se encontró el modelo entrenado en: {trained_weights}')

trained_model = YOLO(str(trained_weights))
base_model = YOLO(BASE_MODEL_ID)

# Predicción con modelo base (sin entrenamiento en tu dataset)
pred_base = base_model.predict(
    source=str(test_image),
    conf=0.25,
    imgsz=640,
    project='runs/predict',
    name='kortxo_base',
    exist_ok=True,
)

# Predicción con modelo entrenado
pred_trained = trained_model.predict(
    source=str(test_image),
    conf=0.25,
    imgsz=640,
    project='runs/predict',
    name='kortxo_trained_full',
    exist_ok=True,
)

print('Imagen de entrada:', test_image)
print('Visualizando resultados directamente desde memoria (sin depender de save_dir).')

# Ultralytics devuelve la imagen anotada en BGR; la convertimos a RGB para matplotlib
base_img = cv2.cvtColor(pred_base[0].plot(), cv2.COLOR_BGR2RGB)
trained_img = cv2.cvtColor(pred_trained[0].plot(), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(base_img)
axes[0].set_title('Modelo base (yolov8n.pt)')
axes[0].axis('off')

axes[1].imshow(trained_img)
axes[1].set_title('Modelo entrenado (best.pt)')
axes[1].axis('off')

plt.tight_layout()
plt.show()
